# IPL Matches EDA - Structured Version

This notebook provides a clearer and more structured analysis of IPL match data from 2008 to 2022, including exploratory analysis and a baseline predictive model for match outcomes.

## 1. Setup and imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

sns.set_theme(style='darkgrid')
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12})

## 2. Load and prepare the data

In [ ]:
project_root = Path.cwd()
matches = pd.read_csv(project_root / 'IPL_Matches_2008_2022.csv')

matches['Date'] = pd.to_datetime(matches['Date'], errors='coerce')
matches['Season'] = matches['Date'].dt.year

rename_map = {
    'Kings XI Punjab': 'Punjab Kings',
    'Delhi Daredevils': 'Delhi Capitals',
    'Rising Pune Supergiants': 'Rising Pune Supergiant'
}
for col in ['Team1', 'Team2', 'WinningTeam', 'TossWinner']:
    matches[col] = matches[col].replace(rename_map)

matches['WinningTeam'] = matches['WinningTeam'].fillna('NoResult')
matches['TossWinner'] = matches['TossWinner'].fillna('NoResult')
matches['Team1Wins'] = (matches['WinningTeam'] == matches['Team1']).astype(int)
matches['TossWinnerIsTeam1'] = (matches['TossWinner'] == matches['Team1']).astype(int)

matches.head()

## 3. Exploratory analysis

In [ ]:
team_counts = pd.concat([matches['Team1'], matches['Team2']]).value_counts().reset_index()
team_counts.columns = ['Team', 'MatchesPlayed']
win_counts = matches['WinningTeam'].value_counts().reset_index()
win_counts.columns = ['Team', 'Wins']
summary = team_counts.merge(win_counts, on='Team', how='left').fillna(0)
summary['WinRatio'] = (summary['Wins'] / summary['MatchesPlayed'] * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=summary.sort_values('MatchesPlayed', ascending=False).head(10), x='MatchesPlayed', y='Team', ax=axes[0], palette='viridis')
axes[0].set_title('Most Active IPL Teams')
axes[0].set_xlabel('Matches Played')
axes[0].set_ylabel('Team')

sns.barplot(data=summary.sort_values('WinRatio', ascending=False).head(10), x='WinRatio', y='Team', ax=axes[1], palette='magma')
axes[1].set_title('Top Win Ratio by Team')
axes[1].set_xlabel('Win Percentage')
axes[1].set_ylabel('Team')
plt.tight_layout()
plt.show()

In [ ]:
toss_decision = matches['TossDecision'].value_counts().reset_index()
toss_decision.columns = ['Decision', 'Count']
sns.barplot(data=toss_decision, x='Decision', y='Count', palette='Set2')
plt.title('Toss Decision Distribution')
plt.show()

## 4. Predictive modeling

In [ ]:
model_df = matches[['Team1', 'Team2', 'Venue', 'TossDecision', 'TossWinnerIsTeam1', 'Team1Wins']].dropna().copy()

X = model_df[['Team1', 'Team2', 'Venue', 'TossDecision', 'TossWinnerIsTeam1']]
y = model_df['Team1Wins']

categorical_features = ['Team1', 'Team2', 'Venue', 'TossDecision']
preprocessor = ColumnTransformer([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
    ('passthrough', 'passthrough', ['TossWinnerIsTeam1'])
])

pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('classifier', LogisticRegression(max_iter=2000, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
pipeline.fit(X_train, y_train)
predictions = pipeline.predict(X_test)

print('Accuracy:', round(accuracy_score(y_test, predictions), 4))
print('Confusion Matrix:')
print(confusion_matrix(y_test, predictions))
print('Classification Report:')
print(classification_report(y_test, predictions, zero_division=0))

## 5. Next steps

- Add more predictive features such as home venue strength or recent team form.
- Compare logistic regression with random forest or XGBoost.
- Create an interactive dashboard for match analysis.